# 0. Imports

In [2]:
# Libraries
import os
import pandas as pd
import numpy as np
import datetime as dt

from datetime import date
from dataclasses import dataclass, field
from dateutil.relativedelta import relativedelta, WE

In [3]:
# Global variables
DIR_PATH = os.getcwd()
DATA_PATH = os.path.join(DIR_PATH, 'market_data_split.csv')

t0 = dt.datetime(2026, 8, 19)

In [4]:
# Data
df = pd.read_csv(DATA_PATH, parse_dates=['maturity']) # SOFR quotes
display(df)

,type,tenor,quote,maturity
0,SOFR,ON,3.620,2026-08-20
1,SR1,Aug-26,96.361,2026-09-01
2,SR1,Sep-26,96.301,2026-10-01
3,SR3,Sep-26,96.195,2026-12-16
4,SR3,Dec-26,96.072,2027-03-17
5,SR3,Mar-27,96.015,2027-06-16
6,SR3,Jun-27,95.999,2027-09-15
7,OIS,2Y,4.000,2028-08-21
8,OIS,3Y,4.000,2029-08-20
9,OIS,5Y,4.040,2031-08-19


# 1. Bootstrapping

## 1.1 Discount curve with exact method

In [5]:
# Date functions

def act360(d1, d2):
    return (d2 - d1).days/360

def act365(d1, d2):
    return (d2 - d1).days/365

def following(d):
    d = np.datetime64(d, 'D')
    return np.busday_offset(d, offsets=0 ,roll='following')

def annual_schedule(t0, n_years):
    delta = relativedelta(years = n_years)
    T = t0 + delta
    return following(T)

In [7]:
# Check sanity of annual_schedule

ois = df[df['type'] == 'OIS']
ois_years = ois["tenor"].str.rstrip("Y").astype(int) 

for n, expected in zip(ois_years, ois["maturity"]):
    sched = annual_schedule(t0, n)
    print(sched)
    

2028-08-21
2029-08-20
2031-08-19
2033-08-19
2036-08-19
2041-08-19
2046-08-20
2056-08-21


In [18]:
from typing import Any


@dataclass
class Instrument:
    type: str
    tenor: str
    quote: float
    start: date
    end: date

def build_instrument(df, t0) -> list[Instrument]:

    insts = []

    for _, row in df.iterrows():

        if row.type == 'SOFR':
            inst = Instrument(row.type, row.tenor, row.quote, t0, row.maturity)

        elif row.type == 'SR1':
            delta = relativedelta(months = -1)
            start = row.maturity + delta
            if start < t0:

                start = t0
            inst = Instrument(row.type, row.tenor, row.quote, start, row.maturity)

        elif row.type == 'SR3':
            delta = relativedelta(months = -3, day = 1, weekday=WE(+3))
            start = row.maturity + delta
            inst = Instrument(row.type, row.tenor, row.quote, start, row.maturity)

        elif row.type == 'OIS':
            tenor_strip = int(row.tenor.rstrip('Y'))
            inst = OIS(row.type, row.tenor, row.quote, t0, row.maturity, 
                       pay_dates=[annual_schedule(t0, i) for i in range(1, tenor_strip + 1)]
                       )

        insts.append(inst)

    return insts

@dataclass
class OIS(Instrument):
    pay_dates: list[date]

class Curve:
    def __init__(self, t0, t=None, logP=None):
        self.t0 = t0
        self.t = np.array([0.0]) if t is None else t # Time to maturity
        self.logP = np.array([0.0]) if logP is None else logP

    def _t(self, d):
        # Get the time to maturity in ACT/365
        d = np.asarray(d, dtype='datetime64[D]')
        t0 = np.datetime64(self.t0, 'D')
        return (d - t0).astype(float) / 365.0

    def P(self, d):
        tq = self._t(d)
        # Check feasible date
        if np.any(tq < 0) or np.any(tq > self.t[-1] + 1e-12):
            raise ValueError("Date is not in range")
        # Interpolation if within range
        return np.exp(np.interp(tq, self.t, self.logP))


    def with_node(self, d, P):
        tn = self._t(d)
        return Curve(self.t0, np.append(self.t, tn), np.append(self.logP, np.log(P)))

    def fwd_simple(self, s, e):
        # Get the delta in ACT /360
        delta = act360(s, e)
        # Compute the forward
        return 1/delta * (self.P(s)/self.P(e) - 1)


def model_quote(inst, curve):
    type = inst.type
    start = inst.start
    end = inst.end

    if type == 'SOFR':
        quote = curve.fwd_simple(start, end)
        
    elif type in ('SR1', 'SR3'):
        quote = (1 - curve.fwd_simple(start, end))

    elif type == 'OIS':
        s = 0
        pay_dates = inst.pay_dates
        pay_dates.insert(0, inst.start)
        for i in range(1, len(pay_dates)):
            delta = act360(pay_dates[i - 1], pay_dates[i])
            P = curve.P(pay_dates[i])
            s += delta * P
        quote = (1 - P) / s

    else:
        raise ValueError("Instrument type not recognized")

    return 100 * quote